# moviepy — Final composite (sub-scene timeline)

Composes the polls-tutorial demo into a single mp4 from:

- per-sub-scene narration: `audio/scene_NN[a|b].mp3` (committed; rendered by `narrator.ipynb`)
- per-sub-scene Manim clips: `media/videos/_manim_render/1080p60/<SceneClass>.mp4` (gitignored; rendered by `manim.ipynb` at -qh)

**Why sub-scenes?** Each Manim animation is paired 1:1 with its own narration mp3. Within a long scene like Models (Scene 4), the `#[model]` macro animation plays during the macro narration, then the ERD animation plays during the ERD/migrations narration. Drift cannot accumulate within a scene because each sub-scene is anchored independently on the global timeline via `with_start(t)`.

13 sub-scenes total: scene 1, 2, 3a, 3b, 4a, 4b, 5a, 5b, 6, 7, 8a, 8b, 9.

`out/` is gitignored — re-run this notebook to regenerate on a fresh checkout.


In [ ]:
from pathlib import Path

import numpy as np

from moviepy import (
    AudioFileClip,
    CompositeAudioClip,
    CompositeVideoClip,
    ImageClip,
    VideoFileClip,
)

# Notebook lives in source/, so paths are relative to that.
AUDIO_DIR = Path("audio")
VIDEO_DIR = Path("media/videos/_manim_render/1080p60")
OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "polls_demo.mp4"

assert AUDIO_DIR.exists(), f"missing {AUDIO_DIR.resolve()}"
assert VIDEO_DIR.exists(), (
    f"missing {VIDEO_DIR.resolve()} — re-render manim.ipynb at -qh first"
)

# Output format — match the Manim 1080p60 input so we don't resample frames.
FRAME_SIZE = (1920, 1080)
FPS = 60

print(f"audio dir: {AUDIO_DIR.resolve()}")
print(f"video dir: {VIDEO_DIR.resolve()}")
print(f"out      : {OUT_PATH.resolve()}")


In [ ]:
# 13 sub-scenes — each pairs ONE narration mp3 with ONE Manim clip.
SCENES = [
    {"id": "1",  "audio": "scene_01.mp3",  "video": "Scene1Opening.mp4"},
    {"id": "2",  "audio": "scene_02.mp3",  "video": "Scene2Reinhardt.mp4"},
    {"id": "3a", "audio": "scene_03a.mp3", "video": "Scene3aTree.mp4"},
    {"id": "3b", "audio": "scene_03b.mp3", "video": "Scene3bMakeTasks.mp4"},
    {"id": "4a", "audio": "scene_04a.mp3", "video": "Scene4aModelMacro.mp4"},
    {"id": "4b", "audio": "scene_04b.mp3", "video": "Scene4bQuestionChoiceERD.mp4"},
    {"id": "5a", "audio": "scene_05a.mp3", "video": "Scene5aServerFnRPC.mp4"},
    {"id": "5b", "audio": "scene_05b.mp3", "video": "Scene5bUseActionCycle.mp4"},
    {"id": "6",  "audio": "scene_06.mp3",  "video": "Scene6aFormMacro.mp4"},
    {"id": "7",  "audio": "scene_07.mp3",  "video": "Scene7aAdminRegister.mp4"},
    {"id": "8a", "audio": "scene_08a.mp3", "video": "Scene8aDIProviders.mp4"},
    {"id": "8b", "audio": "scene_08b.mp3", "video": "Scene8bGuardChain.mp4"},
    {"id": "9",  "audio": "scene_09.mp3",  "video": "Scene9Closing.mp4"},
]

# Tail silence between sub-scenes (seconds). Audio clock advances by
# (audio.duration + TAIL_SILENCE) per sub-scene, giving a natural beat.
TAIL_SILENCE = 0.4

# Compensate for moviepy + ffmpeg's mono->stereo AAC RMS attenuation.
AUDIO_GAIN = 1.5  # ~+3.5 dB


In [ ]:
def build_subscene_segments(scene, scene_start):
    """Return (video_segments, audio_segment, scene_target_duration).

    Video and audio are placed at scene_start on the global timeline. If the
    Manim clip is shorter than the audio + tail, hold the last frame.
    """
    audio = AudioFileClip(str(AUDIO_DIR / scene["audio"]))
    target = audio.duration + TAIL_SILENCE

    vc = VideoFileClip(str(VIDEO_DIR / scene["video"])).without_audio()
    video_segments = [vc.with_start(scene_start)]

    if vc.duration + 1e-3 < target:
        freeze_frame = vc.get_frame(max(0.0, vc.duration - 0.05))
        freeze = (
            ImageClip(freeze_frame)
            .with_duration(target - vc.duration)
            .with_start(scene_start + vc.duration)
        )
        freeze.fps = FPS
        video_segments.append(freeze)
    elif vc.duration > target:
        video_segments[0] = vc.subclipped(0, target).with_start(scene_start)

    audio_segment = audio.with_start(scene_start)
    return video_segments, audio_segment, target


# Pre-flight duration plan, computed from audio (the canonical clock).
print(f"{'SUB':>4}  {'AUDIO':>9}  {'MANIM':>9}  {'OUT':>9}  {'CUMUL':>9}")
total = 0.0
for scene in SCENES:
    a = AudioFileClip(str(AUDIO_DIR / scene["audio"])).duration
    v = VideoFileClip(str(VIDEO_DIR / scene["video"])).duration
    out = a + TAIL_SILENCE
    total += out
    print(f"  {scene['id']:>3}  {a:7.2f}s  {v:7.2f}s  {out:7.2f}s  {total:7.2f}s")
print(f"TOTAL: {total:.2f}s  ({total / 60:.2f} min)")


In [ ]:
# Build all video and audio segments, anchored to the global timeline.
all_video_segments = []
all_audio_segments = []
cursor = 0.0

for scene in SCENES:
    vsegs, aseg, target = build_subscene_segments(scene, cursor)
    all_video_segments.extend(vsegs)
    all_audio_segments.append(aseg)
    cursor += target

TOTAL_DURATION = cursor

final_video = CompositeVideoClip(
    all_video_segments, size=FRAME_SIZE, bg_color=(26, 27, 38)
).with_duration(TOTAL_DURATION)

final_audio = (
    CompositeAudioClip(all_audio_segments)
    .with_volume_scaled(AUDIO_GAIN)
    .with_duration(TOTAL_DURATION)
)
final_audio.fps = 44100

final = final_video.with_audio(final_audio)

print(f"video segments: {len(all_video_segments)}")
print(f"audio segments: {len(all_audio_segments)}")
print(f"total duration: {TOTAL_DURATION:.3f}s ({TOTAL_DURATION / 60:.2f} min)")
print(f"audio gain    : {AUDIO_GAIN}× (~{20 * np.log10(AUDIO_GAIN):.1f} dB)")


In [ ]:
print(f"Writing {OUT_PATH} ...")

final.write_videofile(
    str(OUT_PATH),
    fps=FPS,
    codec="libx264",
    audio_codec="aac",
    audio_bitrate="192k",
    audio_fps=44100,
    threads=4,
    preset="medium",
    ffmpeg_params=["-vsync", "cfr"],
)

print("Done.")


In [ ]:
# Sanity check the final mp4.
clip = VideoFileClip(str(OUT_PATH))
size_mb = OUT_PATH.stat().st_size / 1024 / 1024
print(f"path      : {OUT_PATH}")
print(f"size      : {size_mb:.1f} MB")
print(f"duration  : {clip.duration:.3f}s ({clip.duration / 60:.2f} min)")
print(f"fps       : {clip.fps}")
print(f"resolution: {clip.size}")
print(f"audio?    : {clip.audio is not None}")
if clip.audio is not None:
    print(f"audio fps : {clip.audio.fps}")
    print(f"audio dur : {clip.audio.duration:.3f}s")
    drift = clip.duration - clip.audio.duration
    print(f"a/v drift : {drift * 1000:+.1f} ms")
clip.close()
